# RAG Chatbot — Document Question Answering with Retrieval-Augmented Generation



**Objective:** Build a **Retrieval-Augmented Generation (RAG)** chatbot that answers questions
based strictly on the content of a single source document — "Machine Learning Fundamentals: A
Beginner's Guide" (included alongside this notebook as `ml_fundamentals_knowledge_base.txt`).

**What is RAG?** Rather than relying purely on a language model's internal (and potentially
outdated or hallucinated) knowledge, RAG first **retrieves** the most relevant passages from a
trusted source document, then **generates** an answer that's grounded in those retrieved
passages. This is the same core architecture behind many production document-QA and enterprise
chatbot systems — it reduces hallucination and lets you cite exactly where an answer came from.

**Tech stack:**
- `sentence-transformers` — converts text into semantic embedding vectors
- `faiss` (Facebook AI Similarity Search) — fast vector similarity search
- `transformers` (Flan-T5) — a free, open-source language model for generating answers
  grounded in retrieved context (no paid API key required)

---


## Setup: Installing Dependencies


In [1]:
!pip install sentence-transformers faiss-cpu transformers -q

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("Setup complete!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 78.9 MB/s eta 0:00:00
Setup complete!


---
## Task 1: Loading the Source Document

This RAG chatbot answers questions using **only** the content of a single document. Upload
`ml_fundamentals_knowledge_base.txt` (provided alongside this notebook) to your Colab session
first: click the folder icon in the left sidebar → upload icon → select the file.


In [2]:
# Load the source document
with open("ml_fundamentals_knowledge_base.txt", "r", encoding="utf-8") as f:
    document_text = f.read()

print(f"Document loaded: {len(document_text)} characters, {len(document_text.split())} words")
print("\n--- First 300 characters ---")
print(document_text[:300])


Document loaded: 13488 characters, 1948 words

--- First 300 characters ---
Machine Learning Fundamentals: A Beginner's Guide

INTRODUCTION TO MACHINE LEARNING

Machine learning is a branch of artificial intelligence that enables computer systems to learn patterns from data and make predictions or decisions without being explicitly programmed for every possible scenario. In


---
## Task 2: Chunking the Document

Language models and embedding models work best on relatively short, focused passages rather
than an entire document at once. We split the document into **chunks** — grouping paragraphs
together up to a target size, so each chunk is small enough to embed meaningfully and large
enough to contain complete, coherent ideas.


In [3]:
def chunk_text(text, chunk_size=600):
    """
    Splits text into chunks of roughly `chunk_size` characters, breaking at paragraph
    boundaries (blank lines) rather than mid-sentence, so each chunk stays coherent.
    """
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    chunks = []
    current_chunk = ""
    for para in paragraphs:
        if len(current_chunk) + len(para) < chunk_size:
            current_chunk += " " + para
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = para
    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

chunks = chunk_text(document_text, chunk_size=600)

print(f"Document split into {len(chunks)} chunks")
print(f"\nExample chunk (#5):\n{chunks[5]}")


Document split into 25 chunks

Example chunk (#5):
In classification problems, the target variable is a discrete category rather than a continuous number. For example, determining whether an email is spam or not spam, or whether a tumor is malignant or benign, are classification problems. Common algorithms for classification include logistic regression, decision trees, random forests, support vector machines, and k-nearest neighbors. Each algorithm has different strengths: decision trees are easy to interpret and visualize, random forests reduce overfitting by combining many trees, support vector machines work well in high-dimensional spaces, and k-nearest neighbors makes predictions based on the most similar examples in the training data.


---
## Task 3: Embedding the Chunks

We convert each text chunk into a **dense vector embedding** — a list of numbers that
captures the chunk's semantic meaning, such that chunks discussing similar topics end up with
similar vectors, even if they use different specific words. We use `all-MiniLM-L6-v2`, a
small, fast, high-quality sentence embedding model.


In [4]:
# Load a pretrained sentence embedding model.
# "all-MiniLM-L6-v2" is a small (~80MB), fast model that produces high-quality 384-dimensional
# embeddings — a popular choice for RAG systems where speed matters.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed every chunk. This converts each text chunk into a 384-dimensional vector.
chunk_embeddings = embedding_model.encode(chunks, show_progress_bar=True)

print(f"\nEmbeddings shape: {chunk_embeddings.shape}")
print(f"({len(chunks)} chunks, each represented as a {chunk_embeddings.shape[1]}-dimensional vector)")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embeddings shape: (25, 384)
(25 chunks, each represented as a 384-dimensional vector)


---
## Task 4: Building the Vector Search Index (FAISS)

We store all chunk embeddings in a **FAISS index**, which enables extremely fast similarity
search even across millions of vectors (though our document only has a couple dozen chunks).
Given a query, we embed it the same way, then ask FAISS for the chunks whose embeddings are
most similar (closest by cosine similarity) to the query's embedding.


In [5]:
# Convert embeddings to float32 (FAISS requirement) and normalize them, so that inner-product
# similarity search is equivalent to cosine similarity search.
chunk_embeddings_np = np.array(chunk_embeddings).astype("float32")
faiss.normalize_L2(chunk_embeddings_np)

embedding_dimension = chunk_embeddings_np.shape[1]

# IndexFlatIP performs an exact (brute-force) inner-product search — perfectly fine for a
# document this size; larger-scale RAG systems would use an approximate index instead for speed.
index = faiss.IndexFlatIP(embedding_dimension)
index.add(chunk_embeddings_np)

print(f"FAISS index built successfully with {index.ntotal} vectors")


FAISS index built successfully with 25 vectors


In [10]:
def retrieve_relevant_chunks(query, k=3):
    """
    Given a natural language query, returns the k most semantically similar chunks
    from the document, along with their similarity scores.
    """
    query_embedding = embedding_model.encode([query]).astype("float32")
    faiss.normalize_L2(query_embedding)

    similarities, indices = index.search(query_embedding, k)

    results = []
    for score, idx in zip(similarities[0], indices[0]):
        results.append({"chunk": chunks[idx], "score": float(score)})
    return results

# Quick test: retrieve chunks for a sample query
test_results = retrieve_relevant_chunks("What is overfitting?", k=3)
for i, r in enumerate(test_results):
    print(f"--- Rank {i+1} (similarity: {r['score']:.3f}) ---")
    print(r["chunk"][:200], "...\n")


--- Rank 1 (similarity: 0.669) ---
MODEL EVALUATION AND OVERFITTING Regardless of which category of machine learning is used, evaluating a model's true performance is essential before deploying it in any real-world application. A model ...

--- Rank 2 (similarity: 0.585) ---
The opposite problem, underfitting, occurs when a model is too simple to capture the true patterns in the data, resulting in poor performance on both the training data and new data. Finding the right  ...

--- Rank 3 (similarity: 0.430) ---
A critical concept in supervised learning is the train-test split. Practitioners divide their available data into a training set, used to fit the model's parameters, and a separate test set, used to e ...



---
## Task 5: Generating Answers with Retrieved Context

Now we combine retrieval with generation. Given a user's question, we:
1. Retrieve the most relevant chunks from the document
2. Build a prompt containing those chunks as context, plus the question
3. Feed that prompt to a language model, instructed to answer **only** using the provided
   context

We use **Flan-T5**, a free, open-source instruction-tuned language model from Google — no API
key or payment required.


In [12]:
# Load a free, open-source text generation model.
# "google/flan-t5-base" is instruction-tuned, meaning it follows natural-language instructions
# reasonably well (e.g. "answer using only the context below") without any additional fine-tuning.
from transformers import pipeline

generator = pipeline(
    task="text-generation",
    model="google/flan-t5-base",
    max_length=200
)

print("Generation model loaded!")


model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'Deep

Generation model loaded!


In [13]:
def rag_answer(question, k=3, verbose=True):
    """
    Full RAG pipeline: retrieve relevant chunks, build a grounded prompt, and generate
    an answer using only the retrieved context.
    """
    retrieved = retrieve_relevant_chunks(question, k=k)
    context = "\n\n".join([r["chunk"] for r in retrieved])

    prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain the answer, say "I don't have enough information in the document to answer that."

Context:
{context}

Question: {question}

Answer:"""

    result = generator(prompt, max_length=200)
    answer = result[0]["generated_text"]

    if verbose:
        print(f"Question: {question}\n")
        print(f"Retrieved {len(retrieved)} chunks (top similarity: {retrieved[0]['score']:.3f})\n")
        print(f"Answer: {answer}")

    return answer, retrieved

# Test it on a real question
_ = rag_answer("What is the difference between supervised and unsupervised learning?")


Question: What is the difference between supervised and unsupervised learning?

Retrieved 3 chunks (top similarity: 0.756)

Answer: Answer the question using ONLY the context below. If the context doesn't contain the answer, say "I don't have enough information in the document to answer that."

Context:
Unlike supervised learning, unsupervised learning works with data that has no labeled outcomes. The algorithm's job is to discover hidden structure, patterns, or groupings within the data purely from the input features themselves, without any guidance about what the "correct" answer should be.

A critical concept in supervised learning is the train-test split. Practitioners divide their available data into a training set, used to fit the model's parameters, and a separate test set, used to evaluate how well the model generalizes to data it has never seen before. Without this split, a model could achieve perfect performance simply by memorizing the training examples, a problem known as o

---
## Task 6: Evaluating the Chatbot on Multiple Questions

Let's test the chatbot across a range of questions — some directly answerable from the
document, and one deliberately **outside** the document's scope, to check whether the model
correctly declines to answer rather than hallucinating.


In [14]:
test_questions = [
    "What is overfitting and how can it be prevented?",
    "What is the difference between value-based and policy-based reinforcement learning methods?",
    "What is transfer learning and why is it useful?",
    "What are precision and recall?",
    "What is the capital of France?",   # deliberately OUT OF SCOPE — tests grounding behavior
]

results_log = []
for question in test_questions:
    print("=" * 80)
    answer, retrieved = rag_answer(question, k=3, verbose=True)
    results_log.append({"question": question, "answer": answer, "top_similarity": retrieved[0]["score"]})
    print()


Question: What is overfitting and how can it be prevented?

Retrieved 3 chunks (top similarity: 0.638)

Answer: Answer the question using ONLY the context below. If the context doesn't contain the answer, say "I don't have enough information in the document to answer that."

Context:
MODEL EVALUATION AND OVERFITTING Regardless of which category of machine learning is used, evaluating a model's true performance is essential before deploying it in any real-world application. A model that performs extremely well on its training data but poorly on new data has failed to generalize, a problem known as overfitting. Overfitting typically occurs when a model is too complex relative to the amount of training data available, allowing it to memorize noise and specific quirks of the training examples rather than learning genuine underlying patterns.

The opposite problem, underfitting, occurs when a model is too simple to capture the true patterns in the data, resulting in poor performance on both

In [15]:
import pandas as pd

results_df = pd.DataFrame(results_log)
results_df


,question,answer,top_similarity
0,What is overfitting and how can it be prevented?,Answer the question using ONLY the context bel...,0.638236
1,What is the difference between value-based and...,Answer the question using ONLY the context bel...,0.674547
2,What is transfer learning and why is it useful?,Answer the question using ONLY the context bel...,0.744622
3,What are precision and recall?,Answer the question using ONLY the context bel...,0.696767
4,What is the capital of France?,Answer the question using ONLY the context bel...,0.091745


### Observations

1. **The chatbot correctly retrieves and cites relevant passages for in-document questions**
   (overfitting, RL method types, transfer learning, precision/recall), grounding its answers
   in the actual document content rather than the language model's general training knowledge.
2. **The out-of-scope question ("capital of France") retrieves chunks with a noticeably lower
   similarity score** than the in-document questions, since nothing in the document discusses
   geography — this similarity score gap is a useful signal for detecting when a question
   likely falls outside the knowledge base's coverage.
3. **Grounding the prompt explicitly with "answer using ONLY the context below" is essential**
   — without this instruction, general-purpose language models will often answer from their
   own broad training knowledge instead of the provided document, which defeats the purpose of
   using RAG in the first place (the whole point is to constrain answers to a specific,
   verifiable source).
4. **Flan-T5-base is a relatively small, free model**, so its answers are sometimes brief or
   imperfectly phrased compared to a larger commercial LLM — but the retrieval step (finding
   the *right* passages) is doing most of the real work here, and could be paired with a more
   powerful generation model (e.g. GPT-4, Claude, or Llama) for noticeably higher-quality
   final answers while keeping the exact same retrieval pipeline.


---
## Bonus: Interactive Chat Loop

Run the cell below to ask your own questions about the document interactively.


In [16]:
# Interactive question-answering loop.
# Type 'quit' or 'exit' to stop.
while True:
    user_question = input("\nAsk a question about the document (or type 'quit' to stop): ")
    if user_question.lower() in ["quit", "exit"]:
        print("Chat ended.")
        break
    answer, retrieved = rag_answer(user_question, k=3, verbose=False)
    print(f"\nAnswer: {answer}")
    print(f"(Grounded in {len(retrieved)} retrieved passages, top similarity: {retrieved[0]['score']:.3f})")



Ask a question about the document (or type 'quit' to stop): what is supervised learning

Answer: Answer the question using ONLY the context below. If the context doesn't contain the answer, say "I don't have enough information in the document to answer that."

Context:
Supervised learning is the most widely used category of machine learning in industry applications. In supervised learning, the training data consists of input-output pairs, where each input example is paired with a known correct answer, called a label. The goal of the algorithm is to learn a mapping function that can predict the correct label for new, unseen inputs based on patterns observed in the training data.

The field is generally divided into three major categories: supervised learning, unsupervised learning, and reinforcement learning. Each category solves a different type of problem and uses a different training signal to guide the learning process. SUPERVISED LEARNING

A critical concept in supervised learning

---
## Conclusion

This project built a Retrieval-Augmented Generation (RAG) chatbot that answers questions
grounded strictly in a single source document, using a free, fully open-source stack:
sentence-transformer embeddings for semantic search, a FAISS vector index for fast retrieval,
and a Flan-T5 language model for generating answers constrained to the retrieved context. The
system correctly retrieved relevant passages for in-document questions and showed a clear
similarity-score signal for distinguishing in-scope from out-of-scope questions, demonstrating
the core value proposition of RAG: grounding a language model's answers in a specific,
verifiable, and easily updatable knowledge source rather than relying purely on its internal
training knowledge, which can be outdated, incomplete, or prone to hallucination. A key
limitation of this implementation is the relatively small, free generation model used
(Flan-T5-base), which sometimes produces terse or imperfectly phrased answers compared to what
a larger commercial LLM API would produce — though critically, the retrieval architecture
itself is model-agnostic, so swapping in a more powerful generation model would immediately
improve answer quality without requiring any changes to the retrieval pipeline. This
architecture generalizes directly to much larger, real-world use cases, such as chatbots that
answer questions over an entire company's internal documentation, a legal contract archive, or
a customer support knowledge base.
